In [3]:
import numpy as np

from sklearn.base import (BaseEstimator,TransformerMixin)


class FeatureEngineering(BaseEstimator,TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X = X.copy()

        X['visit_dt'] = pd.to_datetime(
            X['Date'] + ' ' + X['Visit Time'],
            format='%Y-%m-%d %I:%M %p'
        )

        X['hour'] = (
            X['visit_dt']
            .dt.hour
        )

        X['day_of_week'] = (
            X['visit_dt']
            .dt.dayofweek
        )

        X['session'] = np.where(
            X['hour'] < 12,
            'Morning',
            'Evening'
        )

        return X[
            [
                'hour',
                'day_of_week',
                'session'
            ]
        ]

In [4]:
import pickle

with open('clinic_model.pkl', 'rb') as f:
    model = pickle.load(f)

In [5]:
def get_status(count):

    if count >= 5:
        return "Busy"

    elif count >= 2:
        return "Normal"

    return "Free"

In [6]:
import pandas as pd

def predict_clinic_load(date,visit_time,pipeline):

    input_df = pd.DataFrame({
        "Date": [date],
        "Visit Time": [visit_time]
    })

    predicted_count = round(
        pipeline.predict(input_df)[0]
    )

    status = get_status(
        predicted_count
    )

    return {
        "predicted_count": predicted_count,
        "status": status
    }

In [19]:
result = predict_clinic_load(
    date="2027-10-31",
    visit_time="7:00 PM",
    pipeline=model
)

print(result)

{'predicted_count': 1, 'status': 'Free'}
